# Tiny VLA From Scratch — Improved Closed-Loop Control

**AI Builders Lab | TensorFlow | CPU-friendly**

Train a small image-and-language conditioned action policy and watch it control a 2D gripper.

**Changes from v1:** robust visual position features derived from pixels (not simulator ground-truth), expert trajectories and recovery states, an explicit STOP/GRASP phase, slower motion near targets, and rollout success-rate evaluation.

This is a **hybrid visual-perception + learned action policy**, not a pretrained VLA foundation model. The color segmentation is deliberately engineered so students can focus on the language-to-action mapping.

**Run:** Runtime → Run all. Set `COMMAND` to one of six supported instructions.

In [ ]:
import numpy as np
import tensorflow as tf
import matplotlib.pyplot as plt
from matplotlib.animation import FuncAnimation
from matplotlib.patches import Circle, Rectangle
from IPython.display import HTML, display

SEED=17
tf.keras.utils.set_random_seed(SEED)
rng=np.random.default_rng(SEED)
SIZE=64
COLORS={"red":np.array([.95,.20,.20]),"blue":np.array([.15,.40,.95]),"green":np.array([.12,.75,.32])}
NAMES=list(COLORS)
COMMANDS=[f"{verb} the {color} block" for verb in ("touch","pick") for color in NAMES]
VOCAB={w:i+1 for i,w in enumerate(sorted(set(" ".join(COMMANDS).split())))}
STEP=.055
GRASP_RADIUS=.065
TRAIN_EPISODES=1200
EPOCHS=25
print("Supported instructions:", COMMANDS)

## 1. Simulator and camera
The simulator knows exact coordinates; the learned policy **does not receive them**. It receives image-derived object coordinates extracted from the rendered pixels and the language tokens. Objects remain visible when the gripper overlaps them because the gripper is drawn as a thin black outline.

In [ ]:
def scene_new(rng):
    while True:
        blocks=rng.uniform(.15,.85,(3,2)).astype("float32")
        if min(np.linalg.norm(blocks[i]-blocks[j]) for i in range(3) for j in range(i+1,3))>.22:break
    return {"blocks":blocks,"robot":rng.uniform(.1,.9,2).astype("float32"),"holding":-1}

def render(scene):
    img=np.ones((SIZE,SIZE,3),dtype="float32")*.95
    for i,name in enumerate(NAMES):
        x,y=scene["blocks"][i]
        cx,cy=round(x*(SIZE-1)),round((1-y)*(SIZE-1))
        img[max(0,cy-3):min(SIZE,cy+4),max(0,cx-3):min(SIZE,cx+4)]=COLORS[name]
    cx,cy=(scene["robot"]*np.array([SIZE-1,SIZE-1])).round().astype(int)
    cy=SIZE-1-cy
    for ox,oy in [(-4,0),(4,0),(0,-4),(0,4)]:
        xx,yy=np.clip(cx+ox,0,SIZE-1),np.clip(cy+oy,0,SIZE-1)
        img[yy,xx]=[.02,.02,.02]
    return img

def vision_features(image):
    # Simple, inspectable perception: colored-pixel centroids + black gripper markers.
    positions=[]
    for name in NAMES:
        mask=np.max(np.abs(image-COLORS[name]),axis=2)<.08
        yy,xx=np.nonzero(mask)
        if len(xx)==0: raise RuntimeError("Colored block not visible")
        positions.extend([xx.mean()/(SIZE-1),1-yy.mean()/(SIZE-1)])
    dark=np.max(image,axis=2)<.09
    yy,xx=np.nonzero(dark)
    if len(xx)==0: raise RuntimeError("Robot gripper not visible")
    positions.extend([xx.mean()/(SIZE-1),1-yy.mean()/(SIZE-1)])
    return np.array(positions,dtype="float32")

def tokens(command):
    command=command.lower().strip()
    if command not in COMMANDS:raise ValueError(f"Supported commands: {COMMANDS}")
    ids=[VOCAB[w] for w in command.split()]
    return np.array(ids+[0]*(4-len(ids)),dtype="int32")

demo=scene_new(rng)
plt.figure(figsize=(4,4));plt.imshow(render(demo));plt.axis("off");plt.show()
print("Visual features (3 blocks + gripper):",vision_features(render(demo)))

## 2. Expert demonstrations with stop and grasp
The expert outputs three numbers: **dx, dy, close-gripper probability**. Once inside grasp range, the desired movement is **zero**, and `pick` closes the gripper.

Training data includes complete approach trajectories, near-target examples, and perturbations so the network can recover from small mistakes.

In [ ]:
def target_id(command):return next(i for i,name in enumerate(NAMES) if name in command)
def teacher(scene,command):
    target=scene["blocks"][target_id(command)]
    diff=target-scene["robot"]
    distance=np.linalg.norm(diff)
    if distance<GRASP_RADIUS*.78:
        return np.array([0,0,float(command.startswith("pick"))],dtype="float32")
    speed=min(STEP,max(0.,distance-GRASP_RADIUS*.50))
    movement=diff/max(distance,1e-6)*speed/STEP
    return np.array([*movement,0],dtype="float32")

def apply(scene,action,command):
    # The physics/control simulator, not a hidden inference-time expert.
    scene["robot"]=np.clip(scene["robot"]+np.clip(action[:2],-1,1)*STEP,.04,.96)
    i=target_id(command)
    if command.startswith("pick") and action[2]>.5 and np.linalg.norm(scene["robot"]-scene["blocks"][i])<GRASP_RADIUS:
        scene["holding"]=i
    if scene["holding"]>=0:scene["blocks"][scene["holding"]]=scene["robot"].copy()
    return scene

def create_data(n_episodes,seed):
    r=np.random.default_rng(seed)
    visual,words,labels=[],[],[]
    for _ in range(n_episodes):
        scene=scene_new(r)
        cmd=COMMANDS[int(r.integers(len(COMMANDS)))]
        i=target_id(cmd)
        if r.random()<.30:
            scene["robot"]=np.clip(scene["blocks"][i]+r.normal(0,.07,2),.04,.96).astype("float32")
        for t in range(22):
            visual.append(vision_features(render(scene)))
            words.append(tokens(cmd))
            labels.append(teacher(scene,cmd))
            action=teacher(scene,cmd).copy()
            if r.random()<.20:
                action[:2]+=r.normal(0,.14,2) # off-policy recovery states
            apply(scene,action,cmd)
            if scene["holding"]>=0:break
    return np.array(visual),np.array(words),np.array(labels)

v_train,w_train,y_train=create_data(TRAIN_EPISODES,SEED+1)
v_test,w_test,y_test=create_data(150,SEED+2)
print("Train samples:",len(y_train),"Grasp samples:",int((y_train[:,2]>.5).sum()))

## 3. Learn the multimodal action policy
The visual features come from the **camera image**. A learned word embedding represents the instruction. A neural network predicts the motion and grasp action. We weight the grasp loss to counter the class imbalance.

In [ ]:
vis=tf.keras.Input((8,),name="visual")
lang=tf.keras.Input((4,),dtype="int32",name="language")
ve=tf.keras.layers.Dense(64,activation="relu")(vis)
le=tf.keras.layers.Embedding(len(VOCAB)+1,12)(lang)
le=tf.keras.layers.Flatten()(le)
h=tf.keras.layers.Concatenate()([ve,le])
h=tf.keras.layers.Dense(128,activation="relu")(h)
h=tf.keras.layers.Dense(128,activation="relu")(h)
move=tf.keras.layers.Dense(2,activation="tanh",name="move")(h)
grip=tf.keras.layers.Dense(1,activation="sigmoid",name="grip")(h)
model=tf.keras.Model([vis,lang],[move,grip])
model.compile(optimizer=tf.keras.optimizers.Adam(.001),
              loss={"move":"mse","grip":"binary_crossentropy"},
              loss_weights={"move":1,"grip":.8})
hist=model.fit({"visual":v_train,"language":w_train},
               {"move":y_train[:,:2],"grip":y_train[:,2:]},
               validation_data=({"visual":v_test,"language":w_test},
                                {"move":y_test[:,:2],"grip":y_test[:,2:]}),
               epochs=EPOCHS,batch_size=256,verbose=1)
plt.plot(hist.history["loss"],label="Train")
plt.plot(hist.history["val_loss"],label="Validation")
plt.legend();plt.xlabel("Epoch");plt.ylabel("Loss");plt.grid(alpha=.2);plt.show()

## 4. Save and reload the trained policy
The evaluation below uses only the **reloaded neural network** for action predictions. The teacher is never called during inference.

In [ ]:
model.save("tiny_vla_improved.keras")
policy=tf.keras.models.load_model("tiny_vla_improved.keras")
def predict(scene,command):
    visual=vision_features(render(scene))[None,:]
    word=tokens(command)[None,:]
    move,grip=policy([visual,word],training=False)
    return np.r_[move.numpy()[0],grip.numpy()[0]]
print("Reloaded learned policy.")

## 5. Closed-loop control with an explicit arrival/grasp guard
We introduce a **safety/termination guard** based on the camera-derived distance. This prevents perpetual oscillation and ensures the simulated gripper can close at the target. The guard is an engineered control component, **not a learned output**.

The movement toward the target still comes from the learned neural network. You can turn off the guard (`USE_ARRIVAL_GUARD=False`) to inspect raw policy behavior.

In [ ]:
USE_ARRIVAL_GUARD=True
COMMAND="pick the red block"
STEPS=32

def control_action(scene,command):
    action=predict(scene,command)
    if USE_ARRIVAL_GUARD:
        visual=vision_features(render(scene))
        i=target_id(command)
        pos=visual[2*i:2*i+2]
        robot=visual[6:8]
        if np.linalg.norm(pos-robot)<GRASP_RADIUS*.88:
            action[:2]=0
            action[2]=1. if command.startswith("pick") else 0.
    return action

def run_episode(command,seed,record=False):
    s=scene_new(np.random.default_rng(seed))
    frames=[];actions=[]
    for t in range(STEPS):
        if record:frames.append((s["robot"].copy(),s["blocks"].copy(),s["holding"]))
        a=control_action(s,command)
        actions.append(a.copy())
        apply(s,a,command)
        if s["holding"]>=0:break
    if record:frames.append((s["robot"].copy(),s["blocks"].copy(),s["holding"]))
    i=target_id(command)
    distance=np.linalg.norm(s["robot"]-s["blocks"][i])
    success=s["holding"]==i if command.startswith("pick") else distance<GRASP_RADIUS
    return success,distance,frames,actions

success,distance,frames,actions=run_episode(COMMAND,SEED+100,record=True)
print("Command:",COMMAND,"Success:",success,"Final distance:",round(float(distance),3))
fig,ax=plt.subplots(figsize=(5,5))
def draw(k):
    ax.clear();ax.set_xlim(0,1);ax.set_ylim(0,1);ax.set_aspect("equal")
    robot,blocks,holding=frames[k]
    for i,name in enumerate(NAMES):
        ax.add_patch(Rectangle(blocks[i]-.035,.07,.07,color=COLORS[name]))
        ax.text(blocks[i,0],blocks[i,1]+.055,name,ha="center",fontsize=9)
    ax.add_patch(Circle(robot,.025,color="black"))
    if k<len(actions):
        delta=actions[k][:2]*STEP*2
        ax.arrow(*robot,*delta,color="orange",width=.003)
    ax.set_title(f"{COMMAND} | step {k}")
    ax.set_xticks([]);ax.set_yticks([])
animation=FuncAnimation(fig,draw,frames=len(frames),interval=180,repeat=False)
plt.close(fig);display(HTML(animation.to_jshtml()))

## 6. Evaluate success over many random scenes
Report **task completion**, not only training loss. Compare with and without the engineered arrival guard. This reveals how much performance comes from the learned policy versus control logic.

In [ ]:
for guard in [False,True]:
    USE_ARRIVAL_GUARD=guard
    print("\nArrival guard:",guard)
    for cmd in COMMANDS:
        results=[run_episode(cmd,2000+j)[0] for j in range(12)]
        print(f"{cmd:23s} {sum(results):2d}/12")
USE_ARRIVAL_GUARD=True

## Student challenges
1. Compare results with and without the arrival guard.
2. Reduce `TRAIN_EPISODES` from 1200 to 200. How does success change?
3. Change `EPOCHS` from 25 to 8. What changes?
4. Why is camera-based object detection easier here than in real robot images?
5. Which components are **learned**, and which are **hand-engineered**?
6. Why does a general VLA need a pretrained visual-language backbone and many diverse demonstrations?

**Takeaway:** Stable robot behavior depends on data distribution, perception, closed-loop control, and task completion—not simply training epochs.